# Solving Monotone Planar (Rectilinear) 3SAT using Bloons TD

A quick prototype in Python to assess the feasibility of our Unity project.

## Special Definitions
- **Square Grid**: a grid created by drawing all lines of the form `y=c1, x=c2` (for all integers c1 and c2)
- **Square-Grid-aligned Graph**: a graph such that every vertex lie on (c3, c4) for any integer c3 and c4.
  also, every edge consists of a non-empty set of connected line segments of length 1 where each endpoint occurs at (c5, c6) for any integer c5 and c6.
- **Triangle Grid**: a grid created by drawing all lines of the form `y=c10*root(3)/2, y=root(3)*(x-c11), y=root(3)*(x-c12)` for all integers c10, c11, c12.
- **Triangle-Grid-aligned Graph**: a graph such that every vertex lies on some intersection of `y=c13*root(3)/2, y=root(3)*(x-c14), y=root(3)*(x-c15)` for integers c13, c14, and c15. also, every edge consists of a non-empty set of connected line segments of length 1 where each endpoint occurs at some intersection of `y=c16*root(3)/2, y=root(3)*(x-c17), y=root(3)*(x-c18)`, for integers c16, c17, and c18.
- **Triangle-Grid-embedded Graph**: Like triangle-grid-aligned graph, but every edge's line segment set can only have exactly 1 line segment.
  As a consequence, this line segment is always directly connected to two different vertices.

## Important Assessments

- validate MRP3SAT instances, ensuring instances are planar and monotone
- given a valid instance, find a rectilinear embedding/drawing
- align that drawing to a square and triangle grid (SGA-MPR3SAT and TGA-MPR3SAT)
- convert that drawing to planar vertex cover on a triangle grid (TGA-PVC)
- convert the triangle-grid-aligned instance into a triangle-grid embedded instance (TGE-PVC)
- convert the triangle-grid-embedded instance to a Bloons TD instance (BTD) that can be played in a video game

## Implementation Notes

- MRP3SAT is only NP-complete if we allow duplicate variables within clauses
  (or allow clauses with *at most* three variables instead of *exactly* 3). (Theorem 1 of Thai paper)
  - Otherwise, every such instance is satisfiable. A solution can be found in polytime. (Theorem 11 of Pilz paper)
  - We should store variables as multisets, sorted arrays of size 3, or some custom and iterable data structure;
    at minimum, we need to be able to loop through all variables while being able to easily query
    the left-most, middle, and right-most variable in the clause (L, M, R)
- The input format for MRP3SAT should include a certificate of satisfiablilty or state that none exists.
- Our program should pad clauses until they contain exactly 3 variables.
  I recommend repeating the variable with smallest index.
- The grid-like data structures BTD instances should be stored as integers.
  - I propose a map of int to int array: the keys are the y-levels (0 being the variable row),
    and the int array representing x-values.
  - odd y-levels would be drawn +0.5 units to the right during rendering
  - all y-coordinates would be scaled by a factor of root(3)/2 during rendering

## Reduction Notes

- each problem (except BTD) is in NP because they are just restricted versions of 3SAT and VC, which are in NP.
- the only odd numbers we should see is the three nodes in the triangle of the triple-OR gadget for VC.
  everything else (wire gadget, variable gadget) should have an **even** number of nodes.
  **This is crucial for the reduction!!!**
  - the gap between variables should be zero or some even number on the square/triangle grid
  - the number of nodes between the variable node and the triple-OR gadget should be an even number (not counting either endpoint)
- evenness is important for bipartite colouring: both colourings are the same size, so we use up the same number of cover nodes (from k) no matter which bipartition we pick.
- also useful for calculating the limit "k", k = ((total number of nodes) - (3m)) / 2 + (2m)
  - there are m number of triple-OR gadgets, which are all 3-node triangles
  - after subtracting, a bunch of even bipartite trees remain. colouring each takes exactly half of the remainging nodes thanks to evenness.
  - then add back 2 nodes per triangle since it takes at minimum 2 nodes to cover a triangle
- to convert MRP3SAT yes-certificates into BTD yes-certificates, follow the same steps as converting 3SAT certificates into VC certificates.
  Just run an additional BFS to propagate the bipartite colouring from the variable row to the triple-OR gadget.
- the resulting TGE-PVC instance will have max degree of 3. Not sure if this is useful or easily provable.
  We are also trivially guarenteed a max degree of 6 from the triangle grid.


## How To Run

From `python-prototype/`:

```bash
source venv/bin/activate
pip install -r requirements.txt
jupyter notebook mrp3sat_reduction.ipynb
```

Or, without activating the venv:

```bash
venv/bin/python -m pip install -r requirements.txt
venv/bin/jupyter notebook mrp3sat_reduction.ipynb
```

In VS Code, select the interpreter/kernel from `python-prototype/venv` and run cells top to bottom.